# Preprocessing-Pipeline (Beispiel)

Lädt ein einzelnes BraTS-Volumen und visualisiert die T1N- und Seg-Preprocessing-Pipelines Schritt für Schritt — jeweils in drei orthogonalen Ansichten (axial, koronal, sagittal). T1N und Seg werden als separate Figuren gespeichert.

Die Notebooks für die Darstellung wurden größtenteils mit KI erstellt

## Pfade

In [ ]:
import sys
from pathlib import Path

RAW_BRATS_ROOT = Path("/home/sven/Desktop/diffsuion/brain")
DATA_PROC_ROOT = Path.cwd().parent / "data_processing"
OUTPUT_DIR     = Path.cwd() / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

sys.path.insert(0, str(DATA_PROC_ROOT))

TARGET_SIZE = (32, 32, 32)
RNG_SEED    = 42

print(f"raw:    {RAW_BRATS_ROOT}")
print(f"output: {OUTPUT_DIR}")

## Style und Helper

In [ ]:
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle
from matplotlib.colors import ListedColormap

from src.preprocessing import (
    percentile_clip, foreground_bbox, crop_foreground,
    pad_to_cubic, resize_volume, normalize_to_range,
)

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Latin Modern Roman", "CMU Serif", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 10,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

VIEWS = ["axial", "coronal", "sagittal"]

# Diskrete Colormap für die vier BraTS-Seg-Label.
# 0 = Hintergrund (transparent), 1 = NCR, 2 = Ödem, 3 = Enhancing Tumor.
SEG_COLORS = ["#1f77b4", "#ff7f0e", "#d62728"]
SEG_LABEL_NAMES = ["NCR (nekrotischer Kern)", "Ödem", "Enhancing Tumor"]
SEG_CMAP = ListedColormap([(0, 0, 0, 0)] + SEG_COLORS)


def clean_axes(ax):
    ax.set_xticks([])
    ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)


def triplanar(vol):
    """axial, koronal, sagittal — koronal/sagittal um 90° gedreht, damit die
    superior-inferior-Achse vertikal liegt (Kopf oben)."""
    sx, sy, sz = vol.shape
    axial    = vol[:, :, sz // 2]
    coronal  = np.rot90(vol[:, sy // 2, :])
    sagittal = np.rot90(vol[sx // 2, :, :])
    return axial, coronal, sagittal


def auto_norm(img):
    """Percentil-basierte Normalisierung auf [0,1] für die Anzeige in Graustufen."""
    lo, hi = np.percentile(img, [1, 99])
    return np.clip((img - lo) / (hi - lo + 1e-9), 0, 1)


def load_volume(path):
    return nib.load(str(path)).get_fdata().astype(np.float32)


def pick_index(n, seed):
    return int(np.random.default_rng(seed).integers(n))


def bbox_rect_args(bbox, vol_shape, view_idx):
    """Liefert (x, y, w, h) für ein matplotlib-Rectangle, das die BBox auf der
    gewählten Ansicht zeigt. Koronal/sagittal sind durch np.rot90 gedreht.
    bbox ist ein 3-Tupel aus slice-Objekten in (axis_0, axis_1, axis_2)."""
    sx, sy, sz = vol_shape
    if view_idx == 0:   # axial: vol[:, :, sz//2] — image rows=axis 0, cols=axis 1
        return (bbox[1].start, bbox[0].start,
                bbox[1].stop - bbox[1].start,
                bbox[0].stop - bbox[0].start)
    if view_idx == 1:   # koronal: rot90(vol[:, sy//2, :])
        return (bbox[0].start, sz - bbox[2].stop,
                bbox[0].stop - bbox[0].start,
                bbox[2].stop - bbox[2].start)
    # sagittal: rot90(vol[sx//2, :, :])
    return (bbox[1].start, sz - bbox[2].stop,
            bbox[1].stop - bbox[1].start,
            bbox[2].stop - bbox[2].start)


def column_titles(fig, axes, titles_with_shapes, dy=0.008):
    """Spaltentitel auf einer einzigen Baseline plazieren — unabhängig davon,
    dass die Achsen je nach Slice-Aspekt unterschiedlich hoch sind."""
    fig.canvas.draw()
    n_cols = axes.shape[1]
    top_y = max(axes[0, c].get_position().y1 for c in range(n_cols))
    for c, (title, shape) in enumerate(titles_with_shapes):
        bbox_ax = axes[0, c].get_position()
        cx = 0.5 * (bbox_ax.x0 + bbox_ax.x1)
        fig.text(cx, top_y + dy,
                 f"Schritt {c+1}: {title}\nshape {shape}",
                 ha="center", va="bottom", fontsize=9)


def row_labels(axes, labels):
    for r, label in enumerate(labels):
        axes[r, 0].set_ylabel(label, fontsize=10, rotation=0,
                              ha="right", va="center", labelpad=10)

## Beispiel laden

Ein BraTS-Case wird seedbasiert ausgewählt (`RNG_SEED`). Wir laden T1N und die zugehörige Seg-Maske aus dem Roh-Datensatz; die volle Auflösung ist 240×240×155 Voxel.

In [ ]:
cases = sorted(RAW_BRATS_ROOT.glob("BraTS-GLI-*"))
case  = cases[pick_index(len(cases), seed=RNG_SEED)]
t1n_path = next(case.glob("*-t1n.nii.gz"))
seg_path = next(case.glob("*-seg.nii.gz"))

t1n_raw = load_volume(t1n_path)
seg_raw = load_volume(seg_path)

print(f"case: {case.name}")
print(f"t1n: shape={t1n_raw.shape}  range=[{t1n_raw.min():.0f}, {t1n_raw.max():.0f}]")
print(f"seg: shape={seg_raw.shape}  labels={np.unique(seg_raw).astype(int).tolist()}")

## T1N-Pipeline

Folgt `src.preprocessing.preprocess_volume`: **Perzentil-Clip → Crop Foreground → Pad to Cubic → Resize (linear) → Normalize [-1, 1]**. Jede Spalte zeigt einen Pipeline-Schritt, jede Zeile eine orthogonale Ansicht.

In [ ]:
v0 = t1n_raw
v1 = percentile_clip(v0)
v2 = crop_foreground(v1)
v3 = pad_to_cubic(v2)
v4 = resize_volume(v3, TARGET_SIZE)
v5 = normalize_to_range(v4)

t1n_steps = [
    ("Original",                      v0),
    ("Perzentil-Clip 0,5-99,5 %",     v1),
    ("Crop Foreground",               v2),
    ("Pad to Cubic",                  v3),
    (f"Resize {TARGET_SIZE[0]}³ (linear)", v4),
    ("Normalize [-1, 1]",             v5),
]

n_cols = len(t1n_steps)
n_rows = len(VIEWS)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.5 * n_cols, 2.5 * n_rows))

for c, (_, vol) in enumerate(t1n_steps):
    slices = triplanar(vol)
    for r in range(n_rows):
        ax = axes[r, c]
        ax.imshow(auto_norm(slices[r]), cmap="gray", vmin=0, vmax=1)
        clean_axes(ax)

row_labels(axes, VIEWS)
fig.suptitle(f"Preprocessing-Pipeline", fontsize=12, y=0.995)
fig.tight_layout(rect=[0.04, 0, 1, 0.91])
column_titles(fig, axes, [(name, vol.shape) for name, vol in t1n_steps])

fig.savefig(OUTPUT_DIR / "pipeline_t1n.png")
plt.show()